# Torso Decomposition — GPU Search (large-graph #1 push)

**Adrian Ymeri · University of Prishtina · SpOC-3**

This notebook runs the **GPU-parallel** search that the CPU can't: it scores a
whole population of elimination orderings on the GPU each generation, so it
explores orders of magnitude more candidates than the CPU evaluator.

**Pipeline (run top to bottom):**
1. install + upload the project
2. **validate** the GPU evaluator against the official CPU scorer — *bit-for-bit*
3. only if validation passes, run the big GPU search on `large-graph`
4. read off whether it beat the leaderboard target

> **Use a GPU runtime:** Runtime → Change runtime type → **T4 GPU**.
> The validation gate is non-negotiable — if the GPU scorer doesn't exactly
> match `core.evaluate`, the result is meaningless, so we check first.

## 1 · GPU runtime check + dependencies
`numba` (with CUDA) is the GPU evaluator's only extra dependency; Colab ships it.

In [ ]:
!nvidia-smi -L          # must list a GPU (e.g. Tesla T4)
import numba, numpy
from numba import cuda
print("numba", numba.__version__, "| CUDA available:", cuda.is_available())
assert cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then rerun."


## 2 · Upload & unpack the project
Run, then choose the **latest** `torso_project.zip` (the one containing
`gpu_eval.py`).

In [ ]:
import os, zipfile, glob
from google.colab import files
%cd /content
up = files.upload()
z = next(k for k in up if k.endswith('.zip'))
!rm -rf torso_project
with zipfile.ZipFile(z) as zf: zf.extractall('.')
ROOT = os.path.dirname(glob.glob('**/algorithms/continuous/gpu_eval.py', recursive=True)[0]).rsplit('/algorithms',1)[0]
%cd {ROOT}
print('project root:', os.getcwd())
assert os.path.exists('algorithms/continuous/gpu_eval.py'), "old zip — re-upload the one with gpu_eval.py"
print('gpu_eval.py present ✓')


## 3 · VALIDATION GATE — GPU evaluator vs official scorer
This builds a mix of orderings, scores them on the **GPU** and with the **numpy
reference**, and checks the per-step degree sequences and feasibility match
exactly, then cross-checks the reference against `core.evaluate`.

**Do not proceed unless this prints `ALL CHECKS PASSED`** (with the GPU lines
showing `0` mismatches). If it errors, paste the error back — kernels sometimes
need a fix on first real GPU run.

In [ ]:
!python3 tools/validate_gpu.py --problem large-graph --batch 512 --cpu-sample 48


## 4 · The GPU search on large-graph (continues from your best)

This **warm-starts from your banked −5,399,072** (the `data/warmstart_large-graph.json`
distilled from your submissions) — it seeds the archive with your best orderings
and starts CMA-ES from the strongest one, then uses the GPU population to push
*beyond* it. So generation 0 already sits at your current best; every `score`
improvement is genuinely new ground.

`--pop` is the population scored per generation on the GPU (4096 ≈ 3 GB).
The log prints best `score` and `gap` to the leader (−5,493,062) each ~5 s, and
flags `BEAT!` the instant the archive overtakes it. (Add `--cold` to start from
min-degree instead, for comparison.)

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
# auto-loads data/warmstart_large-graph.json -> continues from your banked best
!PYTHONWARNINGS=ignore python3 tools/gpu_search.py --problem large-graph --pop 4096 --budget 1800 --seed 42


## 5 · Push harder (optional)
Different seeds explore different basins; pooling them only helps (best-of-union).
Run a few, then the portfolio pools every `gpucma*` ordering with the rest.

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
for s in (7, 13, 21):
    !PYTHONWARNINGS=ignore python3 tools/gpu_search.py --problem large-graph --pop 4096 --budget 1800 --seed {s} --algo gpucma_s{s}
!python3 tools/portfolio.py --problems large-graph     # pool everything -> portfolio.json


## 6 · Did we beat the leader?
The number that matters is the **Official score** vs the **leaderboard target**
(`-5,493,062`, more negative = better). If `gpu_search` printed
`BEAT THE LEADER`, download the submission and verify it on your machine:

```
from google.colab import files
files.download('submissions/large-graph/gpucma.json')
```

Then on your Mac, drop it into `submissions/large-graph/`, run
`python3 tools/portfolio.py` and `python3 tools/refine_thresholds.py`, and the
pooled portfolio inherits the gain.

**Honest note:** beating the leader requires real headroom on large-graph. The
GPU gives you the *throughput* to look much harder than the CPU could; whether
that's enough to overtake #1 is exactly what this run finds out. Either way the
GPU evaluator (validated bit-for-bit) is the §9 contribution made real.